# Voxae bridge training
Thin wrapper around `voxae.train.train`. Runtime: GPU (T4/L4 for the 2B configs, A100 for 7B).

Locally, build the dataset bundle first — it ships only the referenced images at reduced resolution:

```bash
uv run python scripts/package_for_colab.py --max-px 1536
```

Then upload `data/colab_bundle.zip` to Drive.

In [ ]:
!nvidia-smi
!git clone https://github.com/nhipixel/voxae.git
%cd voxae
!pip install -q -e ".[ml,data]" bitsandbytes peft wandb

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

# Unzip to local disk: Drive is slow for many small reads during training.
!cp /content/drive/MyDrive/voxae/colab_bundle.zip /content/
!unzip -q /content/colab_bundle.zip -d /content/bundle
!ls /content/bundle && wc -l /content/bundle/processed/annotations/*.jsonl

In [ ]:
# Point the configs at the bundle.
import pathlib

import yaml

BUNDLE = "/content/bundle"
JSONL = BUNDLE + "/processed/annotations/voxae_reason.jsonl"
for name in ("smoke_2b", "full_2b"):
    p = pathlib.Path(f"voxae/train/configs/{name}.yaml")
    cfg = yaml.safe_load(p.read_text())
    cfg["data_root"] = BUNDLE
    cfg["train_jsonl"] = JSONL
    cfg["output_dir"] = f"/content/outputs/{name}"
    p.write_text(yaml.safe_dump(cfg, sort_keys=False))
    print(name, "->", cfg["data_root"])

## 1. Overfit smoke test
100 samples. `loss_mask` should fall toward zero — that is the signal the bridge learns at all. If it plateaus high, stop here rather than paying for the full run.

In [ ]:
!python -m voxae.train.train --config voxae/train/configs/smoke_2b.yaml

## 2. Full run
All training samples. Checkpoints resume automatically, so a disconnect costs only the steps since the last save.

In [ ]:
import os

os.environ["WANDB_API_KEY"] = ""  # optional
!python -m voxae.train.train --config voxae/train/configs/full_2b.yaml

In [ ]:
# Persist checkpoints before the session ends.
!mkdir -p /content/drive/MyDrive/voxae/outputs
!cp -r /content/outputs/* /content/drive/MyDrive/voxae/outputs/
!du -sh /content/drive/MyDrive/voxae/outputs/*

In [ ]:
# Quick look at the loss curve.
import json
import pathlib

import matplotlib.pyplot as plt

log = pathlib.Path("/content/outputs/full_2b/train_log.jsonl")
recs = [json.loads(line) for line in log.read_text().splitlines() if line.strip()]
for key in ("loss", "loss_mask", "loss_ce"):
    if key in recs[0]:
        plt.plot([r["step"] for r in recs], [r[key] for r in recs], label=key)
plt.xlabel("step")
plt.ylabel("loss")
plt.legend()
plt.show()

## 3. Evaluation

Scores the trained bridge and the zero-shot baseline on the same split, so the
two land in one comparable table (gIoU/cIoU overall and per query family).
Run this here rather than locally: the trained model needs a GPU to be quick.

The baseline calls a hosted VLM, so it needs an API key; skip that cell if you
only want the trained numbers.

In [ ]:
!python -m voxae.eval.run_eval \
    --split test --predictor trained \
    --checkpoint /content/outputs/full_2b/latest \
    --backbone Qwen/Qwen2-VL-2B-Instruct \
    --data-root /content/bundle --device cuda

In [ ]:
import os

os.environ["VOXAE_VLM_API_KEY"] = ""  # required for the baseline only
!python -m voxae.eval.run_eval \
    --split test --predictor zero-shot \
    --data-root /content/bundle

In [ ]:
# Side-by-side table, ready to paste into the README.
import json
import pathlib

ann = pathlib.Path("/content/bundle/processed/annotations")
candidates = [
    ("trained", ann / "eval_trained_test.json"),
    ("zero-shot", ann / "eval_zero-shot_test.json"),
]
reports = {name: json.loads(p.read_text()) for name, p in candidates if p.exists()}

names = list(reports)
metrics = [k for k in reports[names[0]] if k.startswith(("giou", "ciou"))]
header = " | ".join(names)
print(f"| metric | {header} |")
print("|---|" + "---|" * len(names))
for metric in metrics:
    cells = " | ".join(f"{reports[n].get(metric, 0.0):.4f}" for n in names)
    print(f"| {metric} | {cells} |")